# Kriteria 2 Advanced - Pengembangan dari Notebook Kriteria 1

Notebook ini adalah pengembangan dari notebook referensi Kriteria 1 Advanced.

Tujuan notebook ini:
- mempertahankan seluruh fondasi **Kriteria 1 Advanced**,
- lalu menambahkan komponen yang dibutuhkan untuk memenuhi **Kriteria 2 Advanced**.

Checklist Kriteria 2 yang akan dipenuhi:
- **Basic**: Seq2Seq LSTM Teacher Forcing dengan **Functional API**, membuat ulang satu layer **Dense** dari nol, dan memastikan horizon prediksi **24 step**.
- **Skilled**: membuat Seq2Seq Teacher Forcing dengan **Model Subclassing**, serta menambahkan **Multi-Head Attention** pada baseline LSTM dan Seq2Seq.
- **Advanced**: membuat ulang **Multi-Head Attention** sebagai custom layer dan menggunakannya pada baseline LSTM serta Seq2Seq, lalu menambahkan **custom layer tambahan**.

Catatan belajar:
- bagian yang sudah ada dari Kriteria 1 akan diberi komentar `# [Kriteria 1]`,
- bagian yang baru untuk Kriteria 2 akan diberi komentar `# [Tambahan Kriteria 2 ...]`.


In [ ]:
import importlib.util
import subprocess
import sys

PACKAGE_MAP = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'scikit-learn': 'sklearn',
    'statsmodels': 'statsmodels',
    'tensorflow': 'tensorflow',
}

missing_packages = [pkg for pkg, module_name in PACKAGE_MAP.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
    print('Installed missing packages:', missing_packages)
else:
    print('All required packages are already available.')

import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

SEED = 42
CSV_URL = 'https://drive.google.com/uc?export=download&id=1hpsqSpfjdqIZWqwd259klQSeaNSe5Trr'
ROLLING_WINDOW = 24
HORIZON = 24
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
BATCH_SIZE = 64
EPOCHS = 12
WINDOW_CANDIDATES = (24, 48, 72, 96, 168)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)
print('TensorFlow version:', tf.__version__)


## 1. Pipeline Data dari Kriteria 1 Advanced

Bagian ini hampir sama dengan notebook Kriteria 1 Advanced.

Yang dipertahankan:
- load dataset,
- rolling feature,
- split train/validation/test,
- anti-data-leakage scaling,
- ACF/PACF untuk memilih `window size`,
- baseline `tf.data.Dataset`.

Yang ditambahkan khusus untuk Kriteria 2 Basic:
- membuat **decoder input** untuk **teacher forcing**.


In [ ]:
BASE_FEATURE_COLUMNS = ['Close', 'Volume USDT', 'RSI', 'MACD_Hist', 'ATR', 'KAMAO']
TARGET_COLUMN = 'Close'

# [Kriteria 1] Load dan bersihkan dataset.
def load_dataset(csv_url=CSV_URL):
    df = pd.read_csv(csv_url)
    df.columns = [col.strip() for col in df.columns]
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').drop_duplicates(subset=['Date']).reset_index(drop=True)
    df[BASE_FEATURE_COLUMNS] = df[BASE_FEATURE_COLUMNS].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=BASE_FEATURE_COLUMNS).reset_index(drop=True)
    return df[['Date'] + BASE_FEATURE_COLUMNS].copy()

# [Kriteria 1 Advanced] Tambahkan rolling statistic yang kausal.
def add_rolling_features(df, target_column=TARGET_COLUMN, rolling_window=ROLLING_WINDOW):
    enriched = df.copy()
    enriched['close_roll_mean_24'] = enriched[target_column].rolling(rolling_window, min_periods=rolling_window).mean()
    enriched['close_roll_std_24'] = enriched[target_column].rolling(rolling_window, min_periods=rolling_window).std()
    enriched = enriched.dropna().reset_index(drop=True)
    return enriched

# [Kriteria 1] Split kronologis untuk mencegah leakage.
def chronological_split(df, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO):
    total_ratio = train_ratio + val_ratio + test_ratio
    if not np.isclose(total_ratio, 1.0):
        raise ValueError('Split ratio harus berjumlah 1.0')
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    train_df = df.iloc[:train_end].reset_index(drop=True)
    val_df = df.iloc[train_end:val_end].reset_index(drop=True)
    test_df = df.iloc[val_end:].reset_index(drop=True)
    return train_df, val_df, test_df

# [Kriteria 1] Fit scaler hanya pada data train.
def fit_scalers(train_df, feature_columns):
    scalers = {}
    for col in feature_columns:
        scaler = MinMaxScaler()
        scaler.fit(train_df[[col]])
        scalers[col] = scaler
    return scalers

# [Kriteria 1] Transform split lain dengan scaler dari train.
def transform_with_scalers(df, feature_columns, scalers):
    transformed = df[['Date']].copy()
    for col in feature_columns:
        transformed[col] = scalers[col].transform(df[[col]]).astype(np.float32)
    return transformed

# [Kriteria 1 Advanced] Pilih window size dari analisis lag.
def select_window_size_from_lag_analysis(series, candidate_windows=WINDOW_CANDIDATES):
    max_lag = max(candidate_windows)
    acf_values = acf(series, nlags=max_lag, fft=True)
    pacf_values = pacf(series, nlags=max_lag, method='ywm')
    significance_threshold = 1.96 / np.sqrt(len(series))

    lag_rows = []
    selected_window = 72
    for lag in candidate_windows:
        acf_at_lag = float(acf_values[lag])
        pacf_at_lag = float(pacf_values[lag])
        score = max(abs(acf_at_lag), abs(pacf_at_lag))
        lag_rows.append({
            'lag': lag,
            'acf': acf_at_lag,
            'pacf': pacf_at_lag,
            'score': score,
            'significant': score >= significance_threshold,
        })
        if score >= significance_threshold and selected_window == 72:
            selected_window = lag

    lag_summary = pd.DataFrame(lag_rows)
    return selected_window, significance_threshold, lag_summary

# [Kriteria 1] Bentuk data supervised untuk multi-step forecasting.
def make_supervised_windows(df, feature_columns, target_column, window_size, horizon=HORIZON):
    feature_values = df[feature_columns].to_numpy(dtype=np.float32)
    target_values = df[target_column].to_numpy(dtype=np.float32)
    timestamps = df['Date'].to_numpy()
    target_index = feature_columns.index(target_column)

    X, y, y_timestamps, last_close_values = [], [], [], []
    for end_idx in range(window_size, len(df) - horizon + 1):
        start_idx = end_idx - window_size
        target_slice = slice(end_idx, end_idx + horizon)
        X.append(feature_values[start_idx:end_idx])
        y.append(target_values[target_slice].reshape(horizon, 1))
        y_timestamps.append(timestamps[target_slice])
        last_close_values.append(feature_values[end_idx - 1, target_index])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    y_timestamps = np.asarray(y_timestamps)
    last_close_values = np.asarray(last_close_values, dtype=np.float32)
    return X, y, y_timestamps, last_close_values

# [Tambahan Kriteria 2 Basic] Decoder input teacher forcing dibentuk dari close terakhir + target yang digeser.
def build_decoder_inputs(y_true, last_close_values):
    decoder_inputs = np.zeros_like(y_true, dtype=np.float32)
    decoder_inputs[:, 0, 0] = last_close_values
    decoder_inputs[:, 1:, 0] = y_true[:, :-1, 0]
    return decoder_inputs

# [Kriteria 1 Skilled] Pipeline tf.data.Dataset untuk baseline.
def make_tf_dataset(X, y, batch_size=BATCH_SIZE, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.float32)))
    if shuffle:
        dataset = dataset.shuffle(min(len(X), 2048), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# [Tambahan Kriteria 2 Basic] Pipeline tf.data.Dataset untuk Seq2Seq teacher forcing.
def make_seq2seq_tf_dataset(encoder_inputs, decoder_inputs, targets, batch_size=BATCH_SIZE, shuffle=False):
    inputs = {
        'encoder_inputs': encoder_inputs.astype(np.float32),
        'decoder_inputs': decoder_inputs.astype(np.float32),
    }
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets.astype(np.float32)))
    if shuffle:
        dataset = dataset.shuffle(min(len(encoder_inputs), 2048), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

raw_df = load_dataset(CSV_URL)
feature_engineered_df = add_rolling_features(raw_df)
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + ['close_roll_mean_24', 'close_roll_std_24']

train_raw_df, val_raw_df, test_raw_df = chronological_split(feature_engineered_df)

plt.figure(figsize=(12, 8))
sns.heatmap(train_raw_df[FEATURE_COLUMNS].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Fitur pada Data Train')
plt.show()

decomposition = seasonal_decompose(train_raw_df[TARGET_COLUMN], model='additive', period=24, extrapolate_trend='freq')
decomposition.plot()
plt.suptitle('Dekomposisi Target Close pada Data Train', y=1.02)
plt.show()

selected_window_size, significance_threshold, lag_summary_df = select_window_size_from_lag_analysis(train_raw_df[TARGET_COLUMN])
display(lag_summary_df)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
plot_acf(train_raw_df[TARGET_COLUMN], lags=max(WINDOW_CANDIDATES), ax=axes[0])
plot_pacf(train_raw_df[TARGET_COLUMN], lags=max(WINDOW_CANDIDATES), ax=axes[1], method='ywm')
axes[0].set_title('ACF Close pada Data Train')
axes[1].set_title('PACF Close pada Data Train')
plt.tight_layout()
plt.show()

print(f'Significance threshold: {significance_threshold:.5f}')
print(f'Selected window size from ACF/PACF analysis: {selected_window_size}')

scalers = fit_scalers(train_raw_df, FEATURE_COLUMNS)
train_scaled_df = transform_with_scalers(train_raw_df, FEATURE_COLUMNS, scalers)
val_scaled_df = transform_with_scalers(val_raw_df, FEATURE_COLUMNS, scalers)
test_scaled_df = transform_with_scalers(test_raw_df, FEATURE_COLUMNS, scalers)

X_train, y_train, train_timestamps, train_last_close = make_supervised_windows(train_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)
X_val, y_val, val_timestamps, val_last_close = make_supervised_windows(val_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)
X_test, y_test, test_timestamps, test_last_close = make_supervised_windows(test_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)

train_decoder_inputs = build_decoder_inputs(y_train, train_last_close)
val_decoder_inputs = build_decoder_inputs(y_val, val_last_close)
test_decoder_inputs = build_decoder_inputs(y_test, test_last_close)

train_ds = make_tf_dataset(X_train, y_train, shuffle=True)
val_ds = make_tf_dataset(X_val, y_val, shuffle=False)
test_ds = make_tf_dataset(X_test, y_test, shuffle=False)

train_seq2seq_ds = make_seq2seq_tf_dataset(X_train, train_decoder_inputs, y_train, shuffle=True)
val_seq2seq_ds = make_seq2seq_tf_dataset(X_val, val_decoder_inputs, y_val, shuffle=False)
test_seq2seq_ds = make_seq2seq_tf_dataset(X_test, test_decoder_inputs, y_test, shuffle=False)

print('Feature columns:', FEATURE_COLUMNS)
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('Decoder train shape:', train_decoder_inputs.shape)

baseline_batch_x, baseline_batch_y = next(iter(train_ds))
seq2seq_batch_inputs, seq2seq_batch_y = next(iter(train_seq2seq_ds))
print('Baseline batch shapes:', baseline_batch_x.shape, baseline_batch_y.shape)
print('Seq2Seq batch shapes:', seq2seq_batch_inputs['encoder_inputs'].shape, seq2seq_batch_inputs['decoder_inputs'].shape, seq2seq_batch_y.shape)


## 2. Baseline LSTM dari Kriteria 1

Bagian ini sengaja dipertahankan agar terlihat jelas fondasi dari Kriteria 1.

Nanti pada bagian setelah ini, baseline akan dimodifikasi agar memenuhi Kriteria 2 Skilled dan Advanced.


In [ ]:
# [Kriteria 1] Baseline LSTM sederhana yang sudah kita punya dari notebook sebelumnya.
baseline_lstm_k1_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(selected_window_size, len(FEATURE_COLUMNS))),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(HORIZON),
    tf.keras.layers.Reshape((HORIZON, 1)),
], name='baseline_lstm_k1_model')

baseline_lstm_k1_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mae',
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')],
)

baseline_lstm_k1_model.summary()


## 3. Custom Layer untuk Kriteria 2

Di sinilah pengembangan utama dari Kriteria 1 ke Kriteria 2 terjadi.

Yang ditambahkan:
- `CustomDense` untuk memenuhi Kriteria 2 Basic,
- `CustomMultiHeadAttention` untuk memenuhi Kriteria 2 Advanced,
- `CustomLayerNormalization` sebagai custom layer tambahan untuk memenuhi Kriteria 2 Advanced.

Komentar di dalam kode akan menunjukkan alasan penambahan tiap komponen.


In [ ]:
@tf.keras.utils.register_keras_serializable(package='dltm_ref')
class CustomDense(tf.keras.layers.Layer):
    # [Tambahan Kriteria 2 Basic] Reimplementasi layer Dense dari nol.
    def __init__(self, units, activation=None, use_bias=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.use_bias = use_bias

    def build(self, input_shape):
        last_dim = int(input_shape[-1])
        self.kernel = self.add_weight(
            name='kernel',
            shape=(last_dim, self.units),
            initializer='glorot_uniform',
            trainable=True,
        )
        self.bias = None
        if self.use_bias:
            self.bias = self.add_weight(
                name='bias',
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
            )
        super().build(input_shape)

    def call(self, inputs):
        outputs = tf.linalg.matmul(inputs, self.kernel)
        if self.use_bias:
            outputs = outputs + self.bias
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

    def get_config(self):
        config = super().get_config()
        config.update({
            'units': self.units,
            'activation': tf.keras.activations.serialize(self.activation),
            'use_bias': self.use_bias,
        })
        return config

@tf.keras.utils.register_keras_serializable(package='dltm_ref')
class CustomLayerNormalization(tf.keras.layers.Layer):
    # [Tambahan Kriteria 2 Advanced] Custom layer tambahan: normalization.
    def __init__(self, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.gamma = self.add_weight(
            name='gamma',
            shape=(feature_dim,),
            initializer='ones',
            trainable=True,
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(feature_dim,),
            initializer='zeros',
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        variance = tf.reduce_mean(tf.square(inputs - mean), axis=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return normalized * self.gamma + self.beta

    def get_config(self):
        config = super().get_config()
        config.update({'epsilon': self.epsilon})
        return config

@tf.keras.utils.register_keras_serializable(package='dltm_ref')
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    # [Tambahan Kriteria 2 Advanced] Reimplementasi Multi-Head Attention dari nol.
    def __init__(self, num_heads, key_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        if key_dim <= 0:
            raise ValueError('key_dim must be positive')
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.dropout = dropout
        self.dropout_layer = tf.keras.layers.Dropout(dropout)

    def build(self, input_shape):
        is_multi_input = (
            isinstance(input_shape, (list, tuple))
            and len(input_shape) > 0
            and isinstance(input_shape[0], (list, tuple, tf.TensorShape))
        )
        if is_multi_input:
            query_shape = tf.TensorShape(input_shape[0])
            value_shape = tf.TensorShape(input_shape[1] if len(input_shape) > 1 else input_shape[0])
        else:
            query_shape = tf.TensorShape(input_shape)
            value_shape = tf.TensorShape(input_shape)
        if query_shape.rank is None or value_shape.rank is None or query_shape[-1] is None or value_shape[-1] is None:
            raise ValueError(f'Unable to infer attention dimensions from input_shape={input_shape}')

        query_dim = int(query_shape[-1])
        projection_dim = self.num_heads * self.key_dim
        self.query_dense = CustomDense(projection_dim, name='query_dense')
        self.key_dense = CustomDense(projection_dim, name='key_dense')
        self.value_dense = CustomDense(projection_dim, name='value_dense')
        self.output_dense = CustomDense(query_dim, name='output_dense')

        self.query_dense.build(query_shape)
        self.key_dense.build(value_shape)
        self.value_dense.build(value_shape)
        self.output_dense.build(tf.TensorShape([None, None, projection_dim]))
        super().build(input_shape)

    def _split_heads(self, x):
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]
        x = tf.reshape(x, [batch_size, seq_len, self.num_heads, self.key_dim])
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def _combine_heads(self, x):
        x = tf.transpose(x, perm=[0, 2, 1, 3])
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]
        return tf.reshape(x, [batch_size, seq_len, self.num_heads * self.key_dim])

    def call(self, query, value=None, key=None, training=False):
        if value is None:
            value = query
        if key is None:
            key = value

        query_proj = self._split_heads(self.query_dense(query))
        key_proj = self._split_heads(self.key_dense(key))
        value_proj = self._split_heads(self.value_dense(value))

        attention_logits = tf.matmul(query_proj, key_proj, transpose_b=True)
        attention_logits = attention_logits / tf.math.sqrt(tf.cast(self.key_dim, tf.float32))
        attention_weights = tf.nn.softmax(attention_logits, axis=-1)
        attention_weights = self.dropout_layer(attention_weights, training=training)

        attention_output = tf.matmul(attention_weights, value_proj)
        attention_output = self._combine_heads(attention_output)
        return self.output_dense(attention_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_heads': self.num_heads,
            'key_dim': self.key_dim,
            'dropout': self.dropout,
        })
        return config


## 4. Arsitektur Model untuk Kriteria 2

Bagian ini dibagi menjadi tiga level pengembangan:

1. **Basic**
- membangun Seq2Seq Teacher Forcing dengan **Functional API**,
- menggunakan `CustomDense`.

2. **Skilled**
- membangun Seq2Seq Teacher Forcing dengan **Model Subclassing**,
- menambahkan attention pada baseline dan seq2seq.

3. **Advanced**
- attention yang dipakai bukan built-in, melainkan `CustomMultiHeadAttention`,
- menambahkan `CustomLayerNormalization` sebagai custom layer tambahan.


In [ ]:
# [Tambahan Kriteria 2 Basic] Seq2Seq Teacher Forcing dengan Functional API.
def build_functional_seq2seq(window_size, num_features, horizon, lstm_units=128):
    encoder_inputs = tf.keras.Input(shape=(window_size, num_features), name='encoder_inputs')
    decoder_inputs = tf.keras.Input(shape=(horizon, 1), name='decoder_inputs')

    encoder_outputs, state_h, state_c = tf.keras.layers.LSTM(
        lstm_units,
        return_sequences=True,
        return_state=True,
        name='encoder_lstm',
    )(encoder_inputs)

    decoder_outputs = tf.keras.layers.LSTM(
        lstm_units,
        return_sequences=True,
        name='decoder_lstm',
    )(decoder_inputs, initial_state=[state_h, state_c])

    # [Tambahan Kriteria 2 Basic] Head output memakai CustomDense, bukan Dense built-in.
    outputs = CustomDense(1, name='functional_seq2seq_output')(decoder_outputs)
    return tf.keras.Model([encoder_inputs, decoder_inputs], outputs, name='functional_seq2seq_teacher_forcing')

# [Modifikasi dari Kriteria 1 -> Kriteria 2 Skilled/Advanced]
# Baseline LSTM diubah agar return_sequences=True, lalu ditambah CustomMultiHeadAttention.
def build_attention_baseline_model(window_size, num_features, horizon, lstm_units=128, num_heads=4, key_dim=16, dropout=0.1):
    encoder_inputs = tf.keras.Input(shape=(window_size, num_features), name='encoder_inputs')
    x = tf.keras.layers.LSTM(lstm_units, return_sequences=True, name='baseline_encoder')(encoder_inputs)

    # [Tambahan Kriteria 2 Advanced] Baseline sekarang memakai CustomMultiHeadAttention.
    attention_output = CustomMultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout, name='baseline_self_attention')(x, x, x)

    # [Tambahan Kriteria 2 Advanced] Tambahkan custom normalization setelah attention.
    x = CustomLayerNormalization(name='baseline_attention_norm')(x + attention_output)
    x = tf.keras.layers.LSTM(lstm_units // 2, name='baseline_projection')(x)
    x = CustomDense(64, activation='relu', name='baseline_dense_head')(x)
    x = CustomDense(horizon, name='baseline_output_dense')(x)
    outputs = tf.keras.layers.Reshape((horizon, 1), name='baseline_outputs')(x)
    return tf.keras.Model(encoder_inputs, outputs, name='baseline_lstm_attention_model')

@tf.keras.utils.register_keras_serializable(package='dltm_ref')
class SubclassedSeq2Seq(tf.keras.Model):
    # [Tambahan Kriteria 2 Skilled] Seq2Seq Teacher Forcing dengan Model Subclassing.
    # [Tambahan Kriteria 2 Advanced] Attention yang digunakan adalah CustomMultiHeadAttention.
    def __init__(self, horizon, lstm_units=128, num_heads=4, key_dim=16, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.horizon = horizon
        self.lstm_units = lstm_units
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.dropout_rate = dropout

        self.encoder = tf.keras.layers.LSTM(lstm_units, return_sequences=True, return_state=True, name='encoder_lstm')
        self.decoder = tf.keras.layers.LSTM(lstm_units, return_sequences=True, return_state=True, name='decoder_lstm')
        self.cross_attention = CustomMultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout, name='cross_attention')
        self.norm = CustomLayerNormalization(name='cross_attention_norm')
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.output_dense = CustomDense(1, name='seq2seq_output_dense')

    def call(self, inputs, training=False):
        encoder_inputs, decoder_inputs = inputs
        encoder_outputs, state_h, state_c = self.encoder(encoder_inputs, training=training)
        decoder_outputs, _, _ = self.decoder(decoder_inputs, initial_state=[state_h, state_c], training=training)

        attended = self.cross_attention(decoder_outputs, encoder_outputs, encoder_outputs, training=training)
        x = self.norm(decoder_outputs + attended)
        x = self.dropout(x, training=training)
        return self.output_dense(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            'horizon': self.horizon,
            'lstm_units': self.lstm_units,
            'num_heads': self.num_heads,
            'key_dim': self.key_dim,
            'dropout': self.dropout_rate,
        })
        return config

functional_seq2seq_model = build_functional_seq2seq(selected_window_size, len(FEATURE_COLUMNS), HORIZON)
baseline_attention_model = build_attention_baseline_model(selected_window_size, len(FEATURE_COLUMNS), HORIZON)
subclassed_seq2seq_model = SubclassedSeq2Seq(horizon=HORIZON, name='subclassed_seq2seq_teacher_forcing')

# Build subclassed model dengan sample batch agar summary bisa tampil.
_ = subclassed_seq2seq_model([
    tf.convert_to_tensor(X_train[:2]),
    tf.convert_to_tensor(train_decoder_inputs[:2]),
])

functional_seq2seq_model.compile(optimizer='adam', loss='mae')
baseline_attention_model.compile(optimizer='adam', loss='mae', metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')])
subclassed_seq2seq_model.compile(optimizer='adam', loss='mae')

print('=== Functional Seq2Seq (Kriteria 2 Basic) ===')
functional_seq2seq_model.summary()

print('=== Baseline Attention Model (Kriteria 2 Skilled/Advanced) ===')
baseline_attention_model.summary()

print('=== Subclassed Seq2Seq (Kriteria 2 Skilled/Advanced) ===')
subclassed_seq2seq_model.summary()

# Forward pass untuk memastikan semua model benar-benar menerima shape 24-step.
functional_output = functional_seq2seq_model([
    seq2seq_batch_inputs['encoder_inputs'],
    seq2seq_batch_inputs['decoder_inputs'],
], training=False)
baseline_attention_output = baseline_attention_model(baseline_batch_x, training=False)
subclassed_output = subclassed_seq2seq_model([
    seq2seq_batch_inputs['encoder_inputs'],
    seq2seq_batch_inputs['decoder_inputs'],
], training=False)

print('Functional Seq2Seq output shape:', functional_output.shape)
print('Baseline attention output shape:', baseline_attention_output.shape)
print('Subclassed Seq2Seq output shape:', subclassed_output.shape)


## 5. Training Model Kriteria 2

Di versi sebelumnya notebook ini memang baru berhenti di pembangunan arsitektur.

Agar lebih mudah dipelajari, sekarang kita tambahkan cell training untuk:
- baseline attention model,
- functional seq2seq teacher forcing,
- subclassed seq2seq teacher forcing.

Catatan penting:
- baseline tetap memakai input `X` saja,
- dua model seq2seq memakai pasangan `(encoder_inputs, decoder_inputs)`,
- evaluasi seq2seq di notebook ini masih **teacher forcing evaluation**, belum autoregressive inference.


In [ ]:
# [Tambahan penjelasan belajar] Untuk training model seq2seq, dataset dibuat dalam format tuple.
# Format tuple ini kompatibel untuk Functional API dan Model Subclassing.
def make_seq2seq_tuple_dataset(encoder_inputs, decoder_inputs, targets, batch_size=BATCH_SIZE, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices(((encoder_inputs.astype(np.float32), decoder_inputs.astype(np.float32)), targets.astype(np.float32)))
    if shuffle:
        dataset = dataset.shuffle(min(len(encoder_inputs), 2048), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

train_seq2seq_tuple_ds = make_seq2seq_tuple_dataset(X_train, train_decoder_inputs, y_train, shuffle=True)
val_seq2seq_tuple_ds = make_seq2seq_tuple_dataset(X_val, val_decoder_inputs, y_val, shuffle=False)
test_seq2seq_tuple_ds = make_seq2seq_tuple_dataset(X_test, test_decoder_inputs, y_test, shuffle=False)

common_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

print('Training baseline attention model...')
history_baseline_attention = baseline_attention_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=common_callbacks,
    verbose=1,
)

print('Training functional seq2seq teacher forcing model...')
history_functional_seq2seq = functional_seq2seq_model.fit(
    train_seq2seq_tuple_ds,
    validation_data=val_seq2seq_tuple_ds,
    epochs=EPOCHS,
    callbacks=common_callbacks,
    verbose=1,
)

print('Training subclassed seq2seq teacher forcing model...')
history_subclassed_seq2seq = subclassed_seq2seq_model.fit(
    train_seq2seq_tuple_ds,
    validation_data=val_seq2seq_tuple_ds,
    epochs=EPOCHS,
    callbacks=common_callbacks,
    verbose=1,
)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
axes[0].plot(history_baseline_attention.history['loss'], label='train_loss')
axes[0].plot(history_baseline_attention.history['val_loss'], label='val_loss')
axes[0].set_title('Baseline Attention Training Curve')
axes[0].legend()

axes[1].plot(history_functional_seq2seq.history['loss'], label='train_loss')
axes[1].plot(history_functional_seq2seq.history['val_loss'], label='val_loss')
axes[1].set_title('Functional Seq2Seq Training Curve')
axes[1].legend()

axes[2].plot(history_subclassed_seq2seq.history['loss'], label='train_loss')
axes[2].plot(history_subclassed_seq2seq.history['val_loss'], label='val_loss')
axes[2].set_title('Subclassed Seq2Seq Training Curve')
axes[2].legend()
plt.tight_layout()
plt.show()


## 6. Comparison Hasil Prediksi

Bagian ini dibuat mirip dengan notebook sebelumnya: kita hitung MAE pada data test, lalu tampilkan:
- tabel perbandingan aktual vs prediksi untuk satu sampel horizon 24 step,
- line chart hasil prediksi.

Untuk seq2seq di notebook ini, prediksi test masih menggunakan **teacher forcing decoder input**, karena fokus notebook ini adalah Kriteria 2.


In [ ]:
def evaluate_and_plot_forecast(y_true, y_pred, title):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    print(f'{title} - Test MAE: {mae:.6f}')

    comparison_df = pd.DataFrame({
        'step': np.arange(1, HORIZON + 1),
        'actual_close_scaled': y_true[0, :, 0],
        'predicted_close_scaled': y_pred[0, :, 0],
    })
    comparison_df['difference'] = comparison_df['predicted_close_scaled'] - comparison_df['actual_close_scaled']
    display(comparison_df)

    plt.figure(figsize=(14, 5))
    plt.plot(comparison_df['step'], comparison_df['actual_close_scaled'], marker='o', label='Actual')
    plt.plot(comparison_df['step'], comparison_df['predicted_close_scaled'], marker='x', label='Predicted')
    plt.title(title)
    plt.xlabel('Forecast Step')
    plt.ylabel('Scaled Close')
    plt.legend()
    plt.show()
    return mae

baseline_attention_predictions = baseline_attention_model.predict(X_test, verbose=0)
functional_seq2seq_predictions = functional_seq2seq_model.predict([X_test, test_decoder_inputs], verbose=0)
subclassed_seq2seq_predictions = subclassed_seq2seq_model.predict((X_test, test_decoder_inputs), verbose=0)

baseline_attention_mae = evaluate_and_plot_forecast(
    y_test,
    baseline_attention_predictions,
    'Baseline Attention LSTM - Direct Multi-step Prediction',
)

functional_seq2seq_mae = evaluate_and_plot_forecast(
    y_test,
    functional_seq2seq_predictions,
    'Functional Seq2Seq Teacher Forcing - Test Prediction',
)

subclassed_seq2seq_mae = evaluate_and_plot_forecast(
    y_test,
    subclassed_seq2seq_predictions,
    'Subclassed Seq2Seq Teacher Forcing - Test Prediction',
)

comparison_summary_df = pd.DataFrame({
    'model': [
        'Baseline Attention LSTM',
        'Functional Seq2Seq Teacher Forcing',
        'Subclassed Seq2Seq Teacher Forcing',
    ],
    'test_mae_scaled': [
        baseline_attention_mae,
        functional_seq2seq_mae,
        subclassed_seq2seq_mae,
    ],
})
display(comparison_summary_df.sort_values('test_mae_scaled'))


## 7. Ringkasan Modifikasi dari Kriteria 1 ke Kriteria 2 Advanced

Kalau dibandingkan dengan notebook Kriteria 1, bagian yang harus **ditambahkan atau dimodifikasi** agar memenuhi **Kriteria 2 Advanced** adalah:

1. **Tambahkan decoder input untuk teacher forcing**
- dari Kriteria 1 kita hanya punya `X` dan `y`,
- untuk Kriteria 2 Seq2Seq, kita perlu `decoder_inputs`,
- isinya adalah `last observed Close` pada langkah pertama, lalu `y_true` yang digeser satu langkah.

2. **Tambahkan `CustomDense`**
- baseline Kriteria 1 memakai `Dense` built-in,
- di Kriteria 2 Basic, minimal satu `Dense` harus dibuat ulang dari nol.

3. **Tambahkan model Seq2Seq Teacher Forcing dengan Functional API**
- ini memenuhi level **Basic** pada Kriteria 2.

4. **Ubah baseline LSTM agar bisa menerima attention**
- pada Kriteria 1, baseline cukup `LSTM -> Dense`,
- pada Kriteria 2 Skilled/Advanced, baseline perlu `return_sequences=True` lalu diberi attention.

5. **Tambahkan model Seq2Seq dengan Model Subclassing**
- ini memenuhi level **Skilled** pada Kriteria 2.

6. **Tambahkan `CustomMultiHeadAttention`**
- pada Kriteria 2 Advanced, attention tidak cukup memakai layer built-in,
- kita harus membuat ulang MHA sendiri dan memakainya pada baseline serta seq2seq.

7. **Tambahkan custom layer tambahan**
- di notebook ini dipilih `CustomLayerNormalization`,
- layer ini dipakai sesudah residual connection attention.

Dengan notebook ini, struktur yang terpenuhi adalah:
- **Kriteria 1 Advanced** tetap hidup,
- **Kriteria 2 Basic** hidup lewat `CustomDense` + Functional Seq2Seq,
- **Kriteria 2 Skilled** hidup lewat Subclassed Seq2Seq + attention,
- **Kriteria 2 Advanced** hidup lewat `CustomMultiHeadAttention` + `CustomLayerNormalization`.

Tambahan belajar di versi ini:
- sudah ada cell training,
- sudah ada comparison hasil prediksi seperti notebook sebelumnya,
- tetapi evaluasi seq2seq masih teacher forcing, bukan autoregressive inference.
